# Phase 3 - Unified Model Optimization Notebook

This notebook consolidates the three historical optimization variants into one reproducible workflow while preserving each variant's original intent and search space.


## Optimization Variants Included

- **Approach V1 (Random Search)**: Random sample from a structured grid; geometric-width autoencoder by depth.
- **Approach V2 (Optuna)**: Optuna over latent size, lr, batch size, dropout, depth, hidden width, and one shared activation.
- **Approach V3 (Optuna)**: Optuna over the V2 space plus independent encoder and decoder activations.

Each section below keeps its own configuration dictionary and model construction logic so results remain comparable to the original notebooks.


In [ ]:
# Public Imports
import itertools
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import tensorflow as tf
from optuna.trial import TrialState
from tensorflow import keras
from tensorflow.keras import Sequential, layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Custom Imports
import sys
src_dir = Path("../src")
sys.path.insert(0, str(src_dir))

from data_preprocessing import resolve_project_root
from evaluate import generate_anomaly_metrics_and_threshold
from train import set_global_seed

project_root = resolve_project_root()
set_global_seed(42)

models_dir = project_root / "models"
results_dir = project_root / "results"
models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)


In [ ]:
# Load processed arrays
data_dir = project_root / "data" / "processed"

X_train_path = data_dir / "X_train.npy"
X_test_path = data_dir / "X_test.npy"
y_train_path = data_dir / "y_train.npy"
y_test_path = data_dir / "y_test.npy"

X_train = np.load(X_train_path)
X_test = np.load(X_test_path)
y_train = np.load(y_train_path)
y_test = np.load(y_test_path)

INPUT_DIM = X_train.shape[1]
THRESHOLD_PERCENTILE = 95

print("Data Summary:")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


## Execution Controls

Use the run flags to control computational cost. Keep `RUN_LONG_EXPERIMENTS=False` for smoke tests and set to `True` for full-report replication.


In [ ]:
RUN_LONG_EXPERIMENTS = False

# Default full budgets from original notebooks
V1_N_CONFIGS_FULL = 100
V1_EPOCHS_FULL = 50

V2_N_TRIALS_FULL = 50
V2_TIMEOUT_FULL = 1800

V3_N_TRIALS_FULL = 50
V3_TIMEOUT_FULL = 1800

# Fast smoke-test budgets
V1_N_CONFIGS_SMOKE = 2
V1_EPOCHS_SMOKE = 2

V2_N_TRIALS_SMOKE = 2
V2_TIMEOUT_SMOKE = 120

V3_N_TRIALS_SMOKE = 2
V3_TIMEOUT_SMOKE = 120


## Approach V1 - Random Search Over Structured Grid

This section mirrors the original V1 notebook: sample random configurations from a predefined grid and train each with early stopping.


In [ ]:
# V1 search space
v1_search_space = {
    "latent_dim": [8, 16, 32, 64],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "batch_size": [32, 64, 128],
    "dropout_rate": [0.0, 0.1, 0.2, 0.3],
    "n_hidden_layers": [1, 2, 3],
}

def v1_sample_random_configs(space, n_samples, seed=42):
    """Sample unique random configurations from the V1 grid."""
    rng = random.Random(seed)
    keys = list(space.keys())
    all_combos = list(itertools.product(*space.values()))
    rng.shuffle(all_combos)
    sampled = all_combos[:n_samples]
    return [dict(zip(keys, combo)) for combo in sampled]

def v1_build_autoencoder(input_dim, latent_dim, dropout_rate=0.0, n_hidden_layers=1, activation="relu"):
    """Geometric-width V1 architecture controlled by depth."""
    if n_hidden_layers > 0:
        widths = np.linspace(np.log(input_dim), np.log(max(latent_dim, 2)), n_hidden_layers + 2)
        widths = [int(w) for w in np.exp(widths).astype(int)[1:-1]]
    else:
        widths = []

    model = Sequential(name=f"v1_autoencoder_ld{latent_dim}_L{n_hidden_layers}_do{dropout_rate}")
    model.add(layers.Input(shape=(input_dim,)))

    for width in widths:
        model.add(layers.Dense(width, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(latent_dim, activation=activation, name="bottleneck"))

    for width in reversed(widths):
        model.add(layers.Dense(width, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(input_dim, activation="linear"))
    return model


In [ ]:
def run_v1_random_search():
    n_configs = V1_N_CONFIGS_FULL if RUN_LONG_EXPERIMENTS else V1_N_CONFIGS_SMOKE
    epochs = V1_EPOCHS_FULL if RUN_LONG_EXPERIMENTS else V1_EPOCHS_SMOKE

    configs = v1_sample_random_configs(v1_search_space, n_configs, seed=42)

    X_fit_ae = X_train
    X_val_ae = X_test

    results = []
    best_val_loss = np.inf
    best_model = None
    best_config = None

    for i, config in enumerate(configs):
        start = time.time()
        model = v1_build_autoencoder(
            input_dim=INPUT_DIM,
            latent_dim=config["latent_dim"],
            dropout_rate=config["dropout_rate"],
            n_hidden_layers=config["n_hidden_layers"],
        )
        model.compile(optimizer=Adam(learning_rate=config["learning_rate"]), loss="mse")

        early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

        history = model.fit(
            X_fit_ae,
            X_fit_ae,
            validation_data=(X_val_ae, X_val_ae),
            epochs=epochs,
            batch_size=config["batch_size"],
            callbacks=[early_stop],
            verbose=0,
        )

        val_loss = float(min(history.history["val_loss"]))
        elapsed = time.time() - start
        results.append({**config, "val_loss": val_loss, "train_time_s": elapsed})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model
            best_config = config
            best_model.save(models_dir / "autoencoder_model_best_v1.keras")

    results_df = pd.DataFrame(results).sort_values("val_loss").reset_index(drop=True)
    results_df.to_csv(models_dir / "hyperparam_search_results_v1.csv", index=False)

    print(f"V1 completed with {len(results_df)} configurations.")
    display(results_df.head(10))

    return best_model, best_config, best_val_loss, results_df

v1_best_model, v1_best_config, v1_best_val_loss, v1_results_df = run_v1_random_search()
print("V1 best config:", v1_best_config)
print(f"V1 best validation loss: {v1_best_val_loss:.6f}")


## Approach V2 - Optuna (Shared Activation)

This section mirrors the original V2 notebook and uses Optuna with one shared activation choice for encoder/decoder blocks.


In [ ]:
v2_search_space = {
    "latent_dim": [8, 16, 32, 64, 128],
    "learning_rate": [1e-2, 5e-3, 1e-3, 5e-4, 1e-4],
    "batch_size": [32, 64, 128, 256],
    "dropout_rate": [0.0, 0.1, 0.2, 0.3],
    "n_hidden_layers": [1, 2, 3],
    "hidden_units": [16, 32, 64, 128],
    "activation": ["relu", "tanh", "sigmoid"],
}

def v2_build_autoencoder(input_dim, latent_dim, dropout_rate=0.0, n_hidden_layers=1, hidden_units=32, activation="relu"):
    model = Sequential(name=f"v2_autoencoder_ld{latent_dim}_L{n_hidden_layers}_do{dropout_rate}_hu{hidden_units}_{activation}")
    model.add(layers.Input(shape=(input_dim,)))

    for _ in range(n_hidden_layers):
        model.add(layers.Dense(hidden_units, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(latent_dim, activation=activation, name="bottleneck"))

    for _ in range(n_hidden_layers):
        model.add(layers.Dense(hidden_units, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(input_dim, activation="linear"))
    return model


In [ ]:
def run_v2_optuna():
    n_trials = V2_N_TRIALS_FULL if RUN_LONG_EXPERIMENTS else V2_N_TRIALS_SMOKE
    timeout = V2_TIMEOUT_FULL if RUN_LONG_EXPERIMENTS else V2_TIMEOUT_SMOKE

    X_train_fit, X_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

    def objective(trial):
        config = {
            "latent_dim": trial.suggest_categorical("latent_dim", v2_search_space["latent_dim"]),
            "learning_rate": trial.suggest_categorical("learning_rate", v2_search_space["learning_rate"]),
            "batch_size": trial.suggest_categorical("batch_size", v2_search_space["batch_size"]),
            "dropout_rate": trial.suggest_categorical("dropout_rate", v2_search_space["dropout_rate"]),
            "n_hidden_layers": trial.suggest_categorical("n_hidden_layers", v2_search_space["n_hidden_layers"]),
            "hidden_units": trial.suggest_categorical("hidden_units", v2_search_space["hidden_units"]),
            "activation": trial.suggest_categorical("activation", v2_search_space["activation"]),
        }

        model = v2_build_autoencoder(
            input_dim=INPUT_DIM,
            latent_dim=config["latent_dim"],
            dropout_rate=config["dropout_rate"],
            n_hidden_layers=config["n_hidden_layers"],
            hidden_units=config["hidden_units"],
            activation=config["activation"],
        )

        model.compile(optimizer=Adam(learning_rate=config["learning_rate"]), loss="mse")

        early_stop = EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)
        history = model.fit(
            X_train_fit,
            X_train_fit,
            validation_data=(X_val, X_val),
            epochs=30,
            batch_size=config["batch_size"],
            callbacks=[early_stop],
            verbose=0,
        )

        return float(min(history.history["val_loss"]))

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, timeout=timeout)

    best_config = study.best_params
    best_val_loss = study.best_value

    results_df = study.trials_dataframe()
    results_df.to_csv(models_dir / "hyperparam_search_results_v2.csv", index=False)

    final_model = v2_build_autoencoder(
        input_dim=INPUT_DIM,
        latent_dim=best_config["latent_dim"],
        dropout_rate=best_config["dropout_rate"],
        n_hidden_layers=best_config["n_hidden_layers"],
        hidden_units=best_config["hidden_units"],
        activation=best_config["activation"],
    )
    final_model.compile(optimizer=Adam(learning_rate=best_config["learning_rate"]), loss="mse")

    final_epochs = 100 if RUN_LONG_EXPERIMENTS else 3
    final_model.fit(
        X_train,
        X_train,
        validation_split=0.1,
        epochs=final_epochs,
        batch_size=best_config["batch_size"],
        callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
        verbose=0,
    )

    final_model.save(models_dir / "autoencoder_model_best_v2.keras")

    print("V2 completed.")
    print("Best config:", best_config)
    print(f"Best validation loss: {best_val_loss:.6f}")

    return final_model, best_config, best_val_loss, results_df, study

v2_final_model, v2_best_config, v2_best_val_loss, v2_results_df, v2_study = run_v2_optuna()
display(v2_results_df.head(10))


## Approach V3 - Optuna (Independent Encoder/Decoder Activations)

This section mirrors the original V3 notebook and extends V2 with separate activation selection for encoder and decoder blocks.


In [ ]:
v3_search_space = {
    "latent_dim": [8, 16, 32, 64, 128],
    "learning_rate": [1e-2, 5e-3, 1e-3, 5e-4, 1e-4],
    "batch_size": [32, 64, 128, 256],
    "dropout_rate": [0.0, 0.1, 0.2, 0.3],
    "n_hidden_layers": [1, 2, 3],
    "hidden_units": [16, 32, 64, 128],
    "activation_encoder": ["relu", "tanh"],
    "activation_decoder": ["relu", "sigmoid", "linear"],
}

def v3_build_autoencoder(input_dim, latent_dim, dropout_rate=0.0, n_hidden_layers=1, hidden_units=32, activation_encoder="relu", activation_decoder="relu"):
    model = Sequential(name=f"v3_autoencoder_ld{latent_dim}_L{n_hidden_layers}_do{dropout_rate}_hu{hidden_units}_ae{activation_encoder}_ad{activation_decoder}")
    model.add(layers.Input(shape=(input_dim,)))

    for _ in range(n_hidden_layers):
        model.add(layers.Dense(hidden_units, activation=activation_encoder))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(latent_dim, activation=activation_encoder, name="bottleneck"))

    for _ in range(n_hidden_layers):
        model.add(layers.Dense(hidden_units, activation=activation_decoder))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(input_dim, activation="linear"))
    return model


In [ ]:
def run_v3_optuna():
    n_trials = V3_N_TRIALS_FULL if RUN_LONG_EXPERIMENTS else V3_N_TRIALS_SMOKE
    timeout = V3_TIMEOUT_FULL if RUN_LONG_EXPERIMENTS else V3_TIMEOUT_SMOKE

    X_train_fit, X_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

    def objective(trial):
        config = {
            "latent_dim": trial.suggest_categorical("latent_dim", v3_search_space["latent_dim"]),
            "learning_rate": trial.suggest_categorical("learning_rate", v3_search_space["learning_rate"]),
            "batch_size": trial.suggest_categorical("batch_size", v3_search_space["batch_size"]),
            "dropout_rate": trial.suggest_categorical("dropout_rate", v3_search_space["dropout_rate"]),
            "n_hidden_layers": trial.suggest_categorical("n_hidden_layers", v3_search_space["n_hidden_layers"]),
            "hidden_units": trial.suggest_categorical("hidden_units", v3_search_space["hidden_units"]),
            "activation_encoder": trial.suggest_categorical("activation_encoder", v3_search_space["activation_encoder"]),
            "activation_decoder": trial.suggest_categorical("activation_decoder", v3_search_space["activation_decoder"]),
        }

        model = v3_build_autoencoder(
            input_dim=INPUT_DIM,
            latent_dim=config["latent_dim"],
            dropout_rate=config["dropout_rate"],
            n_hidden_layers=config["n_hidden_layers"],
            hidden_units=config["hidden_units"],
            activation_encoder=config["activation_encoder"],
            activation_decoder=config["activation_decoder"],
        )

        model.compile(optimizer=Adam(learning_rate=config["learning_rate"]), loss="mse")

        early_stop = EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)
        history = model.fit(
            X_train_fit,
            X_train_fit,
            validation_data=(X_val, X_val),
            epochs=30,
            batch_size=config["batch_size"],
            callbacks=[early_stop],
            verbose=0,
        )

        return float(min(history.history["val_loss"]))

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, timeout=timeout)

    best_config = study.best_params
    best_val_loss = study.best_value

    results_df = study.trials_dataframe()
    results_df.to_csv(models_dir / "hyperparam_search_results_v3.csv", index=False)

    final_model = v3_build_autoencoder(
        input_dim=INPUT_DIM,
        latent_dim=best_config["latent_dim"],
        dropout_rate=best_config["dropout_rate"],
        n_hidden_layers=best_config["n_hidden_layers"],
        hidden_units=best_config["hidden_units"],
        activation_encoder=best_config["activation_encoder"],
        activation_decoder=best_config["activation_decoder"],
    )
    final_model.compile(optimizer=Adam(learning_rate=best_config["learning_rate"]), loss="mse")

    final_epochs = 100 if RUN_LONG_EXPERIMENTS else 3
    final_model.fit(
        X_train,
        X_train,
        validation_split=0.1,
        epochs=final_epochs,
        batch_size=best_config["batch_size"],
        callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
        verbose=0,
    )

    final_model.save(models_dir / "autoencoder_model_best_v3.keras")

    print("V3 completed.")
    print("Best config:", best_config)
    print(f"Best validation loss: {best_val_loss:.6f}")

    return final_model, best_config, best_val_loss, results_df, study

v3_final_model, v3_best_config, v3_best_val_loss, v3_results_df, v3_study = run_v3_optuna()
display(v3_results_df.head(10))


## Consolidated Evaluation

Use the same evaluation function and thresholding logic for each best model to keep comparisons fair.


In [ ]:
def evaluate_model(model, label):
    threshold, y_pred, metrics = generate_anomaly_metrics_and_threshold(
        model,
        X_train,
        X_test,
        y_test,
        percentile=THRESHOLD_PERCENTILE,
    )

    print(f"\n{label} Metrics")
    print("-" * 50)
    print(f"Threshold: {threshold:.8f}")
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1-score:  {metrics['f1_score']:.4f}")

    return {
        "approach": label,
        "threshold": threshold,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1_score": metrics["f1_score"],
    }

summary_rows = []

summary_rows.append(evaluate_model(v1_best_model, "V1 Random Search"))
summary_rows.append(evaluate_model(v2_final_model, "V2 Optuna Shared Activation"))
summary_rows.append(evaluate_model(v3_final_model, "V3 Optuna Split Activations"))

summary_df = pd.DataFrame(summary_rows).sort_values("f1_score", ascending=False).reset_index(drop=True)
display(summary_df)
summary_df.to_csv(results_dir / "phase3_approach_comparison.csv", index=False)


## Reproducibility Notes

- Set `RUN_LONG_EXPERIMENTS=True` to reproduce full-budget runs aligned to the original notebooks.
- Keep random seeds fixed (`42`) for comparable trial sampling and train/validation splits.
- Saved artifacts:
  - `models/hyperparam_search_results_v1.csv`
  - `models/hyperparam_search_results_v2.csv`
  - `models/hyperparam_search_results_v3.csv`
  - `models/autoencoder_model_best_v1.keras`
  - `models/autoencoder_model_best_v2.keras`
  - `models/autoencoder_model_best_v3.keras`
  - `results/phase3_approach_comparison.csv`
